# Configuration

In [1]:
import os 

if True ^ os.getcwd().endswith('hte-and-targeting'):
    os.chdir('..')

In [2]:
import pandas as pd 
import numpy as np

In [3]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['text.usetex'] = False

In [4]:
from statsmodels.regression.linear_model import OLS

In [5]:
from core.variables import * 
from core.segment_targeting_estimators import *
from core.dgp import *
from core.segment_targeting_experiments import *
from core.visualization import *

# DGP

Demand model via potential outcome: 
$$
Y_i = \tau_{g(X_i)}T_i + \epsilon_i
$$
where $g: \mathcal{X} \mapsto \mathcal{S}$ denote the segmentation function and $\mathcal{S}$ is the set of discrete segments. 
The expected potential outcome depends on the individual characteristics only through segmentation. 
The following table summarizes the expected potential outcome: 
|           | Treatment 1 | Treatment 2 | ... | Treatment $T$ | Control |
|-----------|:-------------:|:-------------:|:-----:|:-------------:|:-------------:|
| Segment 1 | $a_{11}$    | $a_{12}$      | ... | $a_{1T}$      | $a_{10}$      |
| Segment 2 | $a_{21}$    | $a_{22}$      | ... | $a_{2T}$      | $Y_{20}$      |
| $\vdots$  | $\vdots$    | $\vdots$      | $\ddots$ | $\vdots$      | $\vdots$      |
| Segment $S$ | $a_{S1}$    | $a_{S2}$      | ... | $a_{ST}$      | $a_{S0}$      |

Consumer characteristics are assumed to be standard normal:  
$$
X_i \sim \text{i.i.d. } \mathcal{N}(0, 1)
$$

In [13]:
class MultipleTreatmentsMultipleSegments(object):
    def __init__(
        self, n_segments: int, te_diff: float, segment_func: callable, 
        treatment_sapce: np.ndarray, noise_std: float = 1.0, 
        seed: int = None, **kwargs
    ):
        if seed is not None:
            np.random.seed(seed)

        self.n_segments = n_segments
        self.te_diff = te_diff
        self.segment_te_arr = np.array([(i + 1) * self.te_diff for i in range(self.n_segments)])
        self.segment_func = segment_func
        self.noise_std = noise_std
        self.treatment_sapce = treatment_sapce

    def sample_individuals(self, sample_size: int, seed: int = None) -> np.ndarray:
        """ 
        Sample individuals from the DGP. 
        """
        if seed is not None:
            np.random.seed(seed)
        return np.random.normal(0, 1, size=(sample_size, ))
    
    def sample(self, sample_size, seed: int = None) -> pd.DataFrame:
        if seed is not None:
            np.random.seed(seed)

        # generate individuals for traning data
        covariates = self.sample_individuals(sample_size, seed=seed)  # shape = (sample_size, )
        segments = self.segment_func(covariates)  # shape = (sample_size, )
        consumer_te_arr = self.segment_te_arr[segments]  # shape = (sample_size, )

        # randomly assign treatment
        treatments = np.random.choice(self.treatment_sapce, size=sample_size)

        # calculate outcomes
        epsilon = np.random.normal(0, self.noise_std, size=sample_size)
        outcomes = consumer_te_arr * treatments + epsilon

        return pd.DataFrame({
            'outcome': outcomes, 'treatment': treatments, 'covariates': covariates, 
        })

In [32]:
dgp = MultipleTreatmentsMultipleSegments(
    n_segments=2, te_diff=0.1, segment_func=lambda x: (x >= 0).astype(int), 
    treatment_sapce=np.array([0, 1, 2, 3]), noise_std=1.0, seed=0
)

In [35]:
dgp.sample(1000, seed=0)

,outcome,treatment,covariates
0,0.389094,0,1.764052
1,1.571041,2,0.400157
2,0.273439,3,0.978738
3,0.591790,3,2.240893
4,-0.122619,2,1.867558
...,...,...,...
995,1.714154,2,0.412871
996,-0.229209,2,-0.198399
997,1.375225,0,0.094192
998,-0.354131,2,-1.147611


# Demand Model

In [51]:
class PotentialOutcomeModel(object):
    def __init__(self, n_segments: int, segment_func: callable, treatment_space: np.ndarray) -> None:
        self.segment_func = segment_func 
        self.treatment_space = treatment_space
        self.n_segments = n_segments 

        # placeholder
        self.lift_records = []

    def fit(self, data):
        data = data.copy()
        data['segment'] = self.segment_func(data['covariates'])
        for t in self.treatment_space:
            if t == 0:
                continue
            for s in range(self.n_segments):
                segment_mask = data['segment'] == s
                treatment_mask = (data['treatment'] == t) | (data['treatment'] == 0)

                lift = self.difference_in_mean(
                    treatment_val=t, 
                    treatments=data['treatment'][treatment_mask & segment_mask].values, 
                    outcomes=data['outcome'][treatment_mask & segment_mask].values, 
                )

                self.lift_records.append({'treatment': t, 'segment': s, 'lift': lift})
        
    @staticmethod
    def difference_in_mean(treatment_val: int, treatments: np.ndarray, outcomes: np.ndarray) -> float:
        return outcomes[treatments == treatment_val].mean() - outcomes[treatments == 0].mean()

In [56]:
data = dgp.sample(1000, seed=0)
data['segment'] = dgp.segment_func(data['covariates'])
data

,outcome,treatment,covariates,segment
0,0.389094,0,1.764052,1
1,1.571041,2,0.400157,1
2,0.273439,3,0.978738,1
3,0.591790,3,2.240893,1
4,-0.122619,2,1.867558,1
...,...,...,...,...
995,1.714154,2,0.412871,1
996,-0.229209,2,-0.198399,0
997,1.375225,0,0.094192,1
998,-0.354131,2,-1.147611,0


In [58]:
data.groupby(['treatment', 'segment']).apply(lambda x: x['outcome'].mean())

treatment  segment
0          0         -0.047658
           1          0.109923
1          0          0.052836
           1          0.074869
2          0          0.222610
           1          0.378007
3          0          0.213891
           1          0.632096
dtype: float64

In [59]:
dgp.segment_te_arr

array([0.1, 0.2])

In [53]:
po_model = PotentialOutcomeModel(
    n_segments=dgp.n_segments, 
    segment_func=dgp.segment_func,
    treatment_space=dgp.treatment_sapce
)
po_model.fit(data)

,outcome,treatment,covariates
0,0.389094,0,1.764052
1,1.571041,2,0.400157
2,0.273439,3,0.978738
3,0.591790,3,2.240893
4,-0.122619,2,1.867558
...,...,...,...
995,1.714154,2,0.412871
996,-0.229209,2,-0.198399
997,1.375225,0,0.094192
998,-0.354131,2,-1.147611


In [54]:
po_model.lift_records

[{'treatment': 1, 'segment': 0, 'lift': 0.10049378719139404},
 {'treatment': 1, 'segment': 1, 'lift': -0.03505411309590949},
 {'treatment': 2, 'segment': 0, 'lift': 0.2702681174065962},
 {'treatment': 2, 'segment': 1, 'lift': 0.2680838239487396},
 {'treatment': 3, 'segment': 0, 'lift': 0.2615491957650938},
 {'treatment': 3, 'segment': 1, 'lift': 0.5221731386778437}]